# Data Engineer Take-Home Assessment TEG
This assessment contains multiple tasks simulating a real-world data engineering scenario.
You will be working with two datasets:
- **sales.csv** – sales transactions (contains missing values, inconsistent formats, duplicates)
- **products.json** – product catalog with nested structures

**Tasks:**
1. Ingest & store data
2. Clean & transform data
3. Model data for analytics
4. Write a reporting query
5. Document performance/scalability considerations
6. Build a basic SSRS report
---

## Task 1 – Ingestion & Storage
Load the CSV and JSON files 
Handle missing values and ensure correct schema types.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, TimestampType, DecimalType)
from Data.Schemas.sales_schema import sales_read_schema
from Data.Schemas.products_schema import products_read_schema

spark = SparkSession.builder.appName("teg-pipeline").master("local[*]").getOrCreate()

In [47]:
sales = (spark.read
         .option("header", True)
         .schema(sales_read_schema)
         .csv("Data/Raw/sales.csv"))
sales.show(5) 

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      1|     101.0|2025-08-01|       2| 10.5|
|      2|     102.0|08/02/2025|       1| 20.0|
|      3|      NULL|2025-08-03|       3| NULL|
|      4|     104.0|2025/08/04|       1| 5.75|
|      4|     104.0|2025/08/04|       1| 5.75|
+-------+----------+----------+--------+-----+
only showing top 5 rows


In [48]:
products = (spark.read
            .option("multiLine", True)
            .schema(products_read_schema)
            .json("Data/Raw/products.json"))
products.show(5, truncate=False) 

+----------+----------+-----------+-----------------------------+
|product_id|name      |category   |details                      |
+----------+----------+-----------+-----------------------------+
|101       |Laptop    |Electronics|{BrandA, 1 year, NULL, NULL} |
|102       |Phone     |Electronics|{BrandB, 2 years, NULL, NULL}|
|104       |Desk Chair|Furniture  |{BrandC, NULL, Leather, NULL}|
|105       |Headphones|Electronics|{BrandD, NULL, NULL, true}   |
+----------+----------+-----------+-----------------------------+



## Task 2 – Data Cleaning & Transformation
- Standardize date formats to `YYYY-MM-DD`
- Remove duplicates
- Fill missing `product_id` with `NULL`
- Ensure prices are numeric with two decimal places

In [ ]:
sales_clean = sales.withColumn("sale_date",
    F.coalesce(
        F.try_to_date("sale_date", "yyyy-MM-dd"),
        F.try_to_date("sale_date", "MM/dd/yyyy"),
        F.try_to_date("sale_date", "yyyy/MM/dd"),
    )
).withColumn("price", 
    F.col("price").cast(DecimalType(10, 2)))

sales_clean.printSchema() 
sales_clean.show()

# sales_clean.filter(F.col("sale_date").isNull()).show()   

root
 |-- sale_id: integer (nullable = true)
 |-- product_id: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: decimal(10,2) (nullable = true)

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      1|     101.0|2025-08-01|       2|10.50|
|      2|     102.0|2025-08-02|       1|20.00|
|      3|      NULL|2025-08-03|       3| NULL|
|      4|     104.0|2025-08-04|       1| 5.75|
|      4|     104.0|2025-08-04|       1| 5.75|
|      5|     105.0|2025-08-05|       5| NULL|
+-------+----------+----------+--------+-----+



In [58]:
sales_final = sales_clean.distinct()
sales_final.show()

+-------+----------+----------+--------+-----+
|sale_id|product_id| sale_date|quantity|price|
+-------+----------+----------+--------+-----+
|      4|     104.0|2025-08-04|       1| 5.75|
|      2|     102.0|2025-08-02|       1|20.00|
|      1|     101.0|2025-08-01|       2|10.50|
|      5|     105.0|2025-08-05|       5| NULL|
|      3|      NULL|2025-08-03|       3| NULL|
+-------+----------+----------+--------+-----+



## Task 3 – Data Modeling
Design a star schema for analytics with:
- `fact_sales`
- `dim_product`
- `dim_date`

Provide:
- ER diagram (link or file) - ERD under: Data/ERD/ERD.png
- SQL DDL statements

DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_product (
    product_id    INT           NOT NULL,
    product_name  NVARCHAR(255) NULL,
    category      NVARCHAR(100) NULL,
    brand         NVARCHAR(100) NULL,
    warranty      NVARCHAR(100) NULL,
    material      NVARCHAR(100) NULL,
    wireless      BIT           NULL,
    CONSTRAINT pk_dim_product PRIMARY KEY (product_id)
);

CREATE TABLE dim_date (
    [date]        DATE          NOT NULL,
    [year]        INT           NOT NULL,
    [month]       INT           NOT NULL,
    [day]         INT           NOT NULL,
    month_name    NVARCHAR(20)  NOT NULL,
    day_name      NVARCHAR(20)  NOT NULL,
    is_weekend    BIT           NOT NULL,
    week_of_year  INT           NOT NULL,
    quarter       INT           NOT NULL,
    CONSTRAINT pk_dim_date PRIMARY KEY ([date])
);

CREATE TABLE fact_sales (
    sale_id     INT             NOT NULL,
    product_id  INT             NULL,
    sale_date   DATE            NOT NULL,
    quantity    INT             NULL,
    price       DECIMAL(10, 2)  NULL,
    revenue     DECIMAL(10, 2)  NULL, 
    CONSTRAINT pk_fact_sales PRIMARY KEY (sale_id),
    CONSTRAINT fk_fact_sales_product FOREIGN KEY (product_id) REFERENCES dim_product (product_id),
    CONSTRAINT fk_fact_sales_date    FOREIGN KEY (sale_date)  REFERENCES dim_date ([date])
);

## Task 4 – Reporting Query
Write an SQL query to:
> Show total sales amount and total units sold per product category for the last 30 days.

SELECT COALESCE(p.category, 'Unknown') AS category,
       SUM(f.revenue)  AS total_sales_amount, -- revenue calculated in fact table as quantity * price
       SUM(f.quantity) AS total_units_sold
FROM fact_sales f
LEFT JOIN dim_product p ON f.product_id = p.product_id
WHERE f.sale_date > DATEADD(DAY, -30, (SELECT MAX(sale_date) FROM fact_sales))
GROUP BY COALESCE(p.category, 'Unknown')
ORDER BY total_sales_amount DESC;

## Task 5 – Performance & Scalability
Write a short note explaining how you would:
- Scale the pipeline for 10x data volume
- Make it production-ready (scheduling, monitoring, DQ checks)

### Assumptions
Running on Databricks on AWS and using Airflow to schedule. Data coming into s3. 

### Scaling for 10x data volume
I would run the pipeline on a Databricks job cluster with autoscaling, so it scales up and down with the amount of data. I would store the data as Delta tables, partition by sale date, so queries only read the data they need. Instead of reloading everything, I would only ingest the new file each time and use an append and checkpoints so reruns don't create duplicates and only the new data is added with the ingest run. I would look at the metrics and logs to see how much compute the job is using and scale the cluster up or down if needed.

### Production readiness
**Scheduling:** I would use Airflow to schedule a Databricks job that ingests and transforms the data. It would run once the file lands in the S3 location (using an S3 sensor and running on a schedule for the time the file lands in s3 every day), then trigger the Databricks job to ingest and transform it.

**Monitoring:** I would check the Airflow and Databricks logs and metrics (run time, row counts, compute usage) to see how the job is performing, and scale compute up or down based on that. I would also check the data quality job or notifications to check on the data quality.

**Data quality:** I would create a job that checks the data against the requirements (eg. expected data types, not null, unique keys, valid dates) and alerts if it isn't what is expected. If the issue is serious, such as a null sale_id or duplicate keys, the process would fail so the team gets alerted. If failure isn't required, it would send a notification (for example an email) and the bad rows would be moved to a separate table or included in the notification to review.

## Task 6 – SSRS Reporting Basics
Using the reporting query from Task 4 as the dataset, outline a basic SSRS report.

Provide:
- A parameterised T-SQL stored procedure for the report dataset (e.g. `@StartDate`, `@EndDate`, `@Category`)
- The SSRS dataset and parameter setup (including a default/cascading parameter)
- A sample RDL snippet or description of the Tablix layout, grouping and totals
- Example SSRS expressions for formatting (e.g. currency, conditional colour on a KPI)
- A short note on how you would deploy the report and schedule a subscription

CREATE OR ALTER PROCEDURE dbo.usp_list_categories
AS
BEGIN
    SET NOCOUNT ON;

    SELECT DISTINCT COALESCE(category, 'Unknown') AS category
    FROM dbo.dim_product
    ORDER BY category;
END;
GO

-- Cascading lookup: product names for the selected category
CREATE OR ALTER PROCEDURE dbo.usp_list_product_names
    @Category NVARCHAR(100) = NULL
AS
BEGIN
    SET NOCOUNT ON;

    SELECT DISTINCT product_name
    FROM dbo.dim_product
    WHERE product_name IS NOT NULL
      AND (@Category IS NULL OR COALESCE(category, 'Unknown') = @Category)
    ORDER BY product_name;
END;
GO

CREATE OR ALTER PROCEDURE dbo.usp_sales_per_category
    @StartDate   DATE = NULL,  
    @EndDate     DATE = NULL,
    @Category    NVARCHAR(100) = NULL,
    @ProductName NVARCHAR(255) = NULL   -- cascading: values come from usp_list_product_names (@Category)
AS
BEGIN
    SET NOCOUNT ON;

    IF @EndDate IS NULL
        SELECT @EndDate = MAX(sale_date) FROM dbo.fact_sales;

    IF @StartDate IS NULL
        SET @StartDate = DATEADD(DAY, -30, @EndDate); 

    SELECT
        COALESCE(p.category, 'Unknown') AS category,
        SUM(f.revenue)                  AS total_sales_amount,   -- revenue = quantity * price, calculated in the fact table
        SUM(f.quantity)                 AS total_units_sold
    FROM dbo.fact_sales AS f
    LEFT JOIN dbo.dim_product AS p
        ON f.product_id = p.product_id
    WHERE f.sale_date > @StartDate
      AND f.sale_date <= @EndDate
      AND (@Category IS NULL OR COALESCE(p.category, 'Unknown') = @Category)
      AND (@ProductName IS NULL OR p.product_name = @ProductName)
    GROUP BY COALESCE(p.category, 'Unknown')
    ORDER BY total_sales_amount DESC;
END;
GO

-- to execute above:
-- all:
EXEC dbo.usp_sales_per_category;
-- category filter:
EXEC dbo.usp_sales_per_category @Category = 'Electronics';
-- cascading: category then product name
EXEC dbo.usp_list_product_names @Category = 'Electronics';
EXEC dbo.usp_sales_per_category @Category = 'Electronics', @ProductName = 'Laptop';
-- specific date range and category: 
EXEC dbo.usp_sales_per_category @StartDate = '2025-08-01', @EndDate = '2025-08-05', @Category = 'Furniture';
-- specific date range
EXEC dbo.usp_sales_per_category @StartDate = '2025-08-01', @EndDate = '2025-08-05';

In [ ]:
<?xml version="1.0" encoding="utf-8"?>
<Report xmlns="http://schemas.microsoft.com/sqlserver/reporting/2016/01/reportdefinition">

  <DataSources>
    <DataSource Name="TegDb">
      <ConnectionProperties>
        <DataProvider>SQL</DataProvider>
        <ConnectString>Data Source=localhost;Initial Catalog=teg</ConnectString>
        <IntegratedSecurity>true</IntegratedSecurity>
      </ConnectionProperties>
    </DataSource>
  </DataSources>

  <DataSets>
    <DataSet Name="dsSalesPerCategory">
      <Query>
        <DataSourceName>TegDb</DataSourceName>
        <QueryParameters>
          <QueryParameter Name="@StartDate"><Value>=Parameters!StartDate.Value</Value></QueryParameter>
          <QueryParameter Name="@EndDate"><Value>=Parameters!EndDate.Value</Value></QueryParameter>
          <QueryParameter Name="@Category"><Value>=Parameters!Category.Value</Value></QueryParameter>
        </QueryParameters>
        <CommandType>StoredProcedure</CommandType>
        <CommandText>dbo.usp_sales_per_category</CommandText>
      </Query>
      <Fields>
        <Field Name="category"><DataField>category</DataField></Field>
        <Field Name="total_sales_amount"><DataField>total_sales_amount</DataField></Field>
        <Field Name="total_units_sold"><DataField>total_units_sold</DataField></Field>
      </Fields>
    </DataSet>
  </DataSets>

  <ReportSections>
    <ReportSection>
      <Body>
        <ReportItems>
          <Tablix Name="tblSales">
            <TablixBody>
              <TablixColumns>
                <TablixColumn><Width>3in</Width></TablixColumn>
                <TablixColumn><Width>2.5in</Width></TablixColumn>
                <TablixColumn><Width>2in</Width></TablixColumn>
              </TablixColumns>
              <TablixRows>

                <!-- Header -->
                <TablixRow>
                  <Height>0.3in</Height>
                  <TablixCells>
                    <TablixCell><CellContents><Textbox Name="h1"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>Category</Value><Style><FontWeight>Bold</FontWeight></Style></TextRun></TextRuns><Style /></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="h2"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>Total Sales Amount</Value><Style><FontWeight>Bold</FontWeight></Style></TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="h3"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>Total Units Sold</Value><Style><FontWeight>Bold</FontWeight></Style></TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                  </TablixCells>
                </TablixRow>

                <!-- Data row: one per row returned by the procedure -->
                <TablixRow>
                  <Height>0.3in</Height>
                  <TablixCells>
                    <TablixCell><CellContents><Textbox Name="d1"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>=Fields!category.Value</Value><Style /></TextRun></TextRuns><Style /></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="d2"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>=Fields!total_sales_amount.Value</Value><Style><Format>C2</Format></Style></TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="d3"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>=Fields!total_units_sold.Value</Value><Style><Format>N0</Format></Style></TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                  </TablixCells>
                </TablixRow>

                <!-- Total row: sales total is green if over 50, red otherwise -->
                <TablixRow>
                  <Height>0.3in</Height>
                  <TablixCells>
                    <TablixCell><CellContents><Textbox Name="t1"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>Total</Value><Style><FontWeight>Bold</FontWeight></Style></TextRun></TextRuns><Style /></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="t2"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun>
                        <Value>=Sum(Fields!total_sales_amount.Value)</Value>
                        <Style>
                          <FontWeight>Bold</FontWeight>
                          <Format>C2</Format>
                          <Color>=IIF(IsNothing(Sum(Fields!total_sales_amount.Value)), "Black", IIF(Sum(Fields!total_sales_amount.Value) &gt; 50, "Green", "Red"))</Color>
                        </Style>
                      </TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                    <TablixCell><CellContents><Textbox Name="t3"><CanGrow>true</CanGrow><KeepTogether>true</KeepTogether>
                      <Paragraphs><Paragraph><TextRuns><TextRun><Value>=Sum(Fields!total_units_sold.Value)</Value><Style><FontWeight>Bold</FontWeight><Format>N0</Format></Style></TextRun></TextRuns><Style><TextAlign>Right</TextAlign></Style></Paragraph></Paragraphs><Style /></Textbox></CellContents></TablixCell>
                  </TablixCells>
                </TablixRow>

              </TablixRows>
            </TablixBody>

            <TablixColumnHierarchy>
              <TablixMembers><TablixMember /><TablixMember /><TablixMember /></TablixMembers>
            </TablixColumnHierarchy>

            <TablixRowHierarchy>
              <TablixMembers>
                <TablixMember><KeepWithGroup>After</KeepWithGroup><RepeatOnNewPage>true</RepeatOnNewPage></TablixMember>
                <TablixMember><Group Name="Details" /></TablixMember>
                <TablixMember><KeepWithGroup>Before</KeepWithGroup></TablixMember>
              </TablixMembers>
            </TablixRowHierarchy>

            <DataSetName>dsSalesPerCategory</DataSetName>
            <Top>0in</Top>
            <Left>0in</Left>
            <Height>0.9in</Height>
            <Width>7.5in</Width>
          </Tablix>
        </ReportItems>
        <Height>1in</Height>
      </Body>
      <Width>7.5in</Width>
      <Page />
    </ReportSection>
  </ReportSections>

  <!-- All three are optional: blank = NULL, so the procedure's own defaults apply -->
  <ReportParameters>
    <ReportParameter Name="StartDate">
      <DataType>DateTime</DataType>
      <Nullable>true</Nullable>
      <DefaultValue><Values><Value>=Nothing</Value></Values></DefaultValue>
      <Prompt>Start date (blank = 30 days ending at end date)</Prompt>
    </ReportParameter>
    <ReportParameter Name="EndDate">
      <DataType>DateTime</DataType>
      <Nullable>true</Nullable>
      <DefaultValue><Values><Value>=Nothing</Value></Values></DefaultValue>
      <Prompt>End date (blank = latest sale date)</Prompt>
    </ReportParameter>
    <ReportParameter Name="Category">
      <DataType>String</DataType>
      <Nullable>true</Nullable>
      <DefaultValue><Values><Value>=Nothing</Value></Values></DefaultValue>
      <Prompt>Category (blank = all)</Prompt>
    </ReportParameter>
  </ReportParameters>

  <Language>en-AU</Language>
</Report>

### how to deploy and manage subscription:
deploy by uploading the rdl file 
then go to manage -> subscriptions -> new subscriptions -> add owner and description -> choose deliver (eg. email) -> schedule -> parameters: use default value -> create subscription